# Chapter 09
 Machine Learning for Business Analytics<br>
Concepts, Techniques, and Applications in Python<br>
by Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

Publisher: Wiley; 2nd edition (2024) <br>
<!-- ISBN-13: 978-3031075650 -->

(c) 2024 Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

The code needs to be executed in sequence.

Python packages and Python itself change over time. This can cause warnings or errors.
"Warnings" are for information only and can usually be ignored.
"Errors" will stop execution and need to be fixed in order to get results.

If you come across an issue with the code, please follow these steps

- Check the repository (https://gedeck.github.io/sdsa-code-solutions/) to see if the code has been upgraded. This might solve the problem.
- Report the problem using the issue tracker at https://github.com/gedeck/sdsa-code-solutions/issues
- Paste the error message into Google and see if someone else already found a solution

**Cell 1: Import libraries**  
This cell imports necessary libraries: `matplotlib.pyplot` for plotting, `mlba` for data loading and utility functions, `numpy` and `pandas` for data manipulation, `scipy` for statistical distributions, `xgboost` for gradient boosting, `sklearn.ensemble.RandomForestClassifier` for random forests, `sklearn.metrics` for evaluation metrics, `sklearn.model_selection` for cross‑validation and hyperparameter tuning, and `sklearn.tree` for decision tree models and visualization. The `%matplotlib inline` ensures plots are displayed inline.

In [ ]:
import matplotlib.pyplot as plt
import mlba
import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, train_test_split)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
%matplotlib inline

**Cell 2: Fit a classification tree with max_depth=1 (stump) on RidingMowers data**  
The `RidingMowers` dataset contains two predictors (`Income` and `Lot_Size`) and a binary target `Ownership` (Owner/Nonowner). A decision tree with maximum depth 1 (a stump) is trained. The tree is then visualised using `plot_tree`. The resulting tree will have a single split on one of the predictors. This demonstrates the simplest possible tree and serves as a baseline for understanding how splits are chosen (based on impurity reduction).

In [ ]:
mower_df = mlba.load_data('RidingMowers.csv')
# use max_depth to control tree size (None = full tree)
classTree = DecisionTreeClassifier(random_state=0, max_depth=1)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])
plot_tree(classTree, feature_names=mower_df.columns[:2],
          label='root', class_names=classTree.classes_,
          impurity=False, filled=True, rounded=True)
plt.tight_layout()
plt.show()

**Cell 3: Fit a classification tree with max_depth=2**  
A decision tree with maximum depth 2 is trained on the same data. The tree will have up to 4 leaf nodes. This allows the model to capture slightly more complex interactions between `Income` and `Lot_Size`. The tree is plotted with smaller font size and increased resolution for readability. This illustrates how additional depth can improve discrimination between classes.

In [ ]:
classTree = DecisionTreeClassifier(random_state=0, max_depth=2)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])
plt.figure(figsize=[3.5, 3], dpi=300)
plot_tree(classTree, feature_names=mower_df.columns[:2],
          label='root', class_names=classTree.classes_,
          impurity=False, filled=True, rounded=True, fontsize=7)
plt.tight_layout()
plt.show()

**Cell 4: Fit a full (unpruned) classification tree**  
A decision tree without any depth restriction (`max_depth=None`) is trained on the RidingMowers data. The tree will grow until all leaves are pure or contain very few samples. This often leads to overfitting. The plot shows a larger tree with more splits, visualised with smaller font size and adjusted figure size. This contrasts with the depth‑limited trees and demonstrates the need for pruning or regularisation.

In [ ]:
classTree = DecisionTreeClassifier(random_state=0)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])
plt.figure(figsize=[3.5, 4], dpi=300)
plot_tree(classTree, feature_names=mower_df.columns[:2],
          label='root', class_names=classTree.classes_,
          impurity=False, filled=True, rounded=True, fontsize=7)
plt.tight_layout()
plt.show()

**Cell 5: Text representation of the full tree**  
The `export_text` function prints a textual representation of the decision tree, showing the split conditions, the number of samples at each node, and the class distribution (weights). This is an alternative to graphical visualisation and can be useful for quick inspection or when graphical output is not available.

In [ ]:
from sklearn.tree import export_text
print(export_text(classTree, feature_names=mower_df.columns[:2], show_weights=True))

**Cell 6: Fit a full classification tree on the Universal Bank dataset (with many features)**  
The Universal Bank dataset is loaded, and irrelevant columns (`ID`, `ZIP Code`) are dropped. The target is `Personal Loan`. A decision tree with default parameters (no pruning) is trained on the training set (60% of data). The resulting tree is very large (max depth not limited) and is plotted with a large figure size and high resolution. This tree is likely overfitted, as will be seen in the next cell.

In [ ]:
bank_df = mlba.load_data('UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

X = bank_df.drop(columns=['Personal Loan'])
y = bank_df['Personal Loan']
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

fullClassTree = DecisionTreeClassifier(random_state=1)
fullClassTree.fit(train_X, train_y)
plt.figure(figsize=[22, 13], dpi=300)
plot_tree(fullClassTree, feature_names=train_X.columns, label='root',
          impurity=False, filled=True, rounded=True, fontsize=10)
plt.tight_layout()
plt.show()

**Cell 7: Performance of the full tree on training and holdout sets**  
Confusion matrices and accuracy are printed for both the training set and the holdout set. The training accuracy is likely very high (close to 100%), while holdout accuracy is lower, indicating overfitting. This demonstrates the need for pruning or regularisation.

In [ ]:
mlba.classificationSummary(y_true=train_y, y_pred=fullClassTree.predict(train_X))
mlba.classificationSummary(y_true=holdout_y, y_pred=fullClassTree.predict(holdout_X))

**Cell 8: Cross‑validation accuracy of a default tree**  
A decision tree with default parameters is evaluated using 5‑fold cross‑validation on the training set. The accuracy scores for each fold are printed. This gives a more robust estimate of expected performance than a single train‑test split.

In [ ]:
treeClassifier = DecisionTreeClassifier(random_state=1)

scores = cross_val_score(treeClassifier, train_X, train_y, cv=5)
print('Accuracy scores of each fold: ', [f'{acc:.3f}' for acc in scores])

**Cell 9: Fit a pruned (regularised) decision tree**  
A decision tree with constraints (`max_depth=30`, `min_samples_split=20`, `min_impurity_decrease=0.01`) is trained. These parameters limit the growth of the tree, reducing overfitting. The tree is plotted, and it will be visibly smaller than the unpruned tree. The choice of parameters is somewhat arbitrary; the next cells will optimise them systematically.

In [ ]:
smallClassTree = DecisionTreeClassifier(max_depth=30, min_samples_split=20,
                        min_impurity_decrease=0.01, random_state=1)
smallClassTree.fit(train_X, train_y)

plot_tree(smallClassTree, feature_names=train_X.columns, label='root',
          impurity=False, filled=True, rounded=True, fontsize=10)
plt.tight_layout()
plt.show()

**Cell 10: Performance of the pruned tree**  
Confusion matrices and accuracy are computed for the pruned tree on both training and holdout sets. The holdout accuracy should be higher than that of the unpruned tree, indicating better generalisation, while training accuracy may be slightly lower.

In [ ]:
mlba.classificationSummary(y_true=train_y, y_pred=smallClassTree.predict(train_X))
mlba.classificationSummary(y_true=holdout_y, y_pred=smallClassTree.predict(holdout_X))

**Cell 11: Hyperparameter tuning via GridSearchCV**  
A two‑stage grid search is performed to find optimal hyperparameters for the decision tree. The first coarse grid explores larger ranges; the second finer grid focuses on promising regions. The best parameters and corresponding cross‑validated accuracy are printed. The `best_estimator_` is stored for later evaluation. This systematic tuning often yields a model that generalises well.

In [ ]:
# Start with an initial guess for parameters
param_grid = {
    'max_depth': [10, 20, 30, 40],
    'min_samples_split': [20, 40, 60, 80, 100],
    'min_impurity_decrease': [0, 0.0005, 0.001, 0.005, 0.01],
}
gridSearch = GridSearchCV(DecisionTreeClassifier(random_state=1), param_grid, cv=5,
                          n_jobs=-1)  # n_jobs=-1 will utilize all available CPUs
gridSearch.fit(train_X, train_y)
print(f'Initial score: {gridSearch.best_score_:.4f}')
print('Initial parameters: ', gridSearch.best_params_)

# Adapt grid based on result from initial grid search
param_grid = {
    'max_depth': list(range(2, 16)),  # 14 values
    'min_samples_split': list(range(10, 22)), # 11 values
    'min_impurity_decrease': [0.0003, 0.0005, 0.0007], # 3 values
}
gridSearch = GridSearchCV(DecisionTreeClassifier(random_state=1), param_grid, cv=5,
                          n_jobs=-1)
gridSearch.fit(train_X, train_y)
print(f'Improved score: {gridSearch.best_score_:.4f}')
print('Improved parameters: ', gridSearch.best_params_)

bestClassTree = gridSearch.best_estimator_

 Evaluating performance

**Cell 12: Training set performance of tuned tree**  
Confusion matrix and accuracy for the best tree on the training set. This shows how well the model fits the training data after tuning.

In [ ]:
# fine-tuned tree: training
mlba.classificationSummary(y_true=train_y, y_pred=bestClassTree.predict(train_X))

**Cell 13: Holdout performance of tuned tree**  
Confusion matrix and accuracy on the holdout set. This is the final estimate of generalisation performance. The tuned tree should achieve better holdout accuracy than the unpruned tree, demonstrating the benefit of hyperparameter optimisation.

In [ ]:
# fine-tuned tree: holdout
mlba.classificationSummary(y_true=holdout_y, y_pred=bestClassTree.predict(holdout_X))

 Plotting tree

**Cell 14: Visualise the tuned decision tree**  
The best tree found by grid search is plotted. The tree is of moderate size, reflecting the regularisation constraints (e.g., `max_depth` likely around 8–10). The plot shows split conditions, class distributions, and leaf predictions.

In [ ]:
plt.figure(figsize=[8.5, 6], dpi=300)
plot_tree(bestClassTree, feature_names=train_X.columns, label='root',
          impurity=False, filled=True, rounded=True, fontsize=8)
plt.tight_layout()
plt.show()

**Cell 15: Regression tree for Toyota Corolla prices (randomised search)**  
The Toyota Corolla dataset is loaded (first 1000 rows). Predictors include `Age`, `KM`, `Fuel_Type` (one‑hot encoded), etc., with target `Price`. A `RandomizedSearchCV` is used to tune a `DecisionTreeRegressor` over a distribution of hyperparameters (`max_depth`, `min_impurity_decrease`, `min_samples_split`). The best parameters are printed, and the model is evaluated on training and holdout sets using regression metrics (RMSE, MAE, MAPE, etc.). This demonstrates hyperparameter tuning for regression tasks.

In [ ]:
toyotaCorolla_df = mlba.load_data('ToyotaCorolla.csv').iloc[:1000,:]
toyotaCorolla_df = toyotaCorolla_df.rename(columns={'Age_08_04': 'Age', 'Quarterly_Tax': 'Tax'})

predictors = ['Age', 'KM', 'Fuel_Type', 'HP', 'Met_Color', 'Automatic', 'CC',
              'Doors', 'Tax', 'Weight']
outcome = 'Price'

X = pd.get_dummies(toyotaCorolla_df[predictors], drop_first=True)
y = toyotaCorolla_df[outcome]

train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

# user randomized search to find optimized tree
param_dist = {
    'max_depth': list(range(1, 25)),
    'min_impurity_decrease': scipy.stats.expon(scale=1),
    'min_samples_split': list(range(2, 50)),
}
randSearch = RandomizedSearchCV(DecisionTreeRegressor(), param_dist, cv=5,
                                n_jobs=-1, n_iter=100, random_state=1)
randSearch.fit(train_X, train_y)
print('Best parameters: ', randSearch.best_params_)

regTree = randSearch.best_estimator_

print(pd.DataFrame({
    'training': mlba.regressionMetrics(y_true=train_y, y_pred=regTree.predict(train_X)),
    'holdout': mlba.regressionMetrics(y_true=holdout_y, y_pred=regTree.predict(holdout_X)),
}).transpose().round(3))

**Cell 16: Visualise the regression tree**  
The tuned regression tree is plotted. The tree structure shows splits on predictors like `Age`, `KM`, `HP`, etc., with leaf nodes containing predicted average prices. The tree is relatively large but still interpretable.

In [ ]:
plt.figure(figsize=[13, 5], dpi=300)
plot_tree(regTree, feature_names=train_X.columns, label='root',
          impurity=False, filled=True, rounded=True, fontsize=4)
plt.tight_layout()
plt.show()


The figure shown in the book was created using the `export_graphviz` function from the `sklearn.tree` module. It requires the `graphviz` package to be installed. We wrapped it into the `mlba.plotDecisionTree` function. There are other packages to visualize trees, such as `dtreeviz`.

**Cell 17: Alternative tree visualisation (if Graphviz is available)**  
This cell attempts to produce a more polished tree diagram using `mlba.plotDecisionTree`, which uses Graphviz. It also saves the output to a PDF. If the required packages are not installed, an exception is caught and a message is printed.

In [ ]:
try:
    display(mlba.plotDecisionTree(regTree, feature_names=train_X.columns,
                                rotate=True, pdfFile=figures_dir / 'RT-tree-toyota-rotate.pdf'))
except Exception:
    print('The tree could not be plotted, install the required packages.')

**Cell 18: Random forest for classification (Universal Bank)**  
A random forest classifier with 500 trees is trained on the Universal Bank dataset. The feature importances (mean decrease in impurity) and their standard deviations across trees are computed. The features are sorted by importance and printed. This helps identify which predictors are most influential for predicting personal loan acceptance.

In [ ]:
bank_df = mlba.load_data('UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

X = bank_df.drop(columns=['Personal Loan'])
y = bank_df['Personal Loan']
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4, random_state=1)

rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf.fit(train_X, train_y)

# variable (feature) importance plot
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

df = pd.DataFrame({'feature': train_X.columns, 'importance': importances, 'std': std})
df = df.sort_values('importance')
print(df)

**Cell 19: Horizontal bar plot of feature importances**  
A horizontal bar chart is created showing the feature importances with error bars representing the standard deviation across trees. This visualisation makes it easy to compare the relative importance of predictors. `Income` and `CCAvg` (credit card average spending) are typically the most important.

In [ ]:
ax = df.plot(kind='barh', xerr='std', x='feature', legend=False)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Cell 20: Confusion matrix for random forest on holdout set**  
The holdout set predictions from the random forest are evaluated, producing a confusion matrix and accuracy. Random forests typically achieve high accuracy and generalise well due to the ensemble averaging.

In [ ]:
# confusion matrix for holdout set
mlba.classificationSummary(y_true=holdout_y, y_pred=rf.predict(holdout_X))

**Cell 21: XGBoost classifier (default parameters)**  
An XGBoost classifier is trained with default hyperparameters on the Universal Bank dataset. The holdout confusion matrix and accuracy are printed. XGBoost often outperforms single trees and even random forests on many tabular datasets.

In [ ]:
boost = xgb.XGBClassifier()
boost.fit(train_X, train_y)
mlba.classificationSummary(y_true=holdout_y, y_pred=boost.predict(holdout_X))

**Cell 22: ROC curves for decision tree, random forest, and XGBoost**  
A function `plot_roc_curve` is defined to compute and plot the ROC curve for a given model. The ROC curves for the best tuned decision tree, random forest, and XGBoost are plotted on the same axes, with two subplots: a full view and a zoom‑in on the high‑sensitivity region. The area under the curve (AUC) is displayed in the legend. This allows direct comparison of model discrimination performance. XGBoost typically achieves the highest AUC.

In [ ]:
def plot_roc_curve(model, X, y, model_name, pos_label, *, color, axes):
    fpr, tpr, _ = roc_curve(y, model.predict_proba(X)[:, 1], pos_label=pos_label)
    roc_auc = auc(fpr, tpr)
    for ax in axes:
        ax.plot(fpr, tpr, color=color, label=f'{model_name} (AUC = {roc_auc:0.3f})')
        ax.plot([0, 1], [0, 1], color='grey', linestyle='--')
        ax.set_xlabel('False positive rate (1 - specificity)')
        ax.set_ylabel('True positive rate (sensitivity)')
        ax.legend(loc="lower right")
    axes[0].set_xlim([-0.02, 1.02])
    axes[0].set_ylim([-0.02, 1.02])
    axes[1].set_xlim([-0.004, 0.2])
    axes[1].set_ylim([0.8, 1.004])
    return ax

fig, axes = plt.subplots(ncols=2, figsize=[10, 5])
plot_roc_curve(bestClassTree, holdout_X, holdout_y, 'Decision tree', 1, color='gray',
               axes=axes)
plot_roc_curve(rf, holdout_X, holdout_y, 'Random forest', 1, color='blue', axes=axes)
plot_roc_curve(boost, holdout_X, holdout_y, 'xgboost', 1, color='red', axes=axes)
plt.tight_layout()
plt.show()

**Cell 23: XGBoost with class weighting (focused on minority class)**  
An XGBoost model is trained with `scale_pos_weight=10` to give more weight to the positive class (loan acceptors). This is a common technique for imbalanced classification. The confusion matrix and ROC curve are compared with the default XGBoost model. The focused model typically improves recall (sensitivity) at the cost of some precision, shifting the ROC curve leftwards in the high‑sensitivity region.

In [ ]:
boost_focused = xgb.XGBClassifier(scale_pos_weight=10)
boost_focused.fit(train_X, train_y)
mlba.classificationSummary(y_true=holdout_y, y_pred=boost_focused.predict(holdout_X))

fig, axes = plt.subplots(ncols=2, figsize=[10, 5])
plot_roc_curve(boost, holdout_X, holdout_y, 'xgboost', 1, color='red', axes=axes)
plot_roc_curve(boost_focused, holdout_X, holdout_y, 'xgboost (focused)', 1,
               color='green', axes=axes)
plt.tight_layout()
plt.show()


  Figures demonstrating construction of a classification tree

**Cell 24: Base scatter plot of RidingMowers data**  
A helper function `basePlot` creates a scatter plot of the RidingMowers data with owners and non‑owners shown in different colours. This is used as the background for subsequent figures that illustrate how decision tree splits partition the feature space.

In [ ]:
def basePlot(ax):
    mower_df.loc[mower_df.Ownership=='Owner'].plot(x='Income', y='Lot_Size', style='o',
                                                markerfacecolor='C0', markeredgecolor='C0',
                                                ax=ax, zorder=100)
    mower_df.loc[mower_df.Ownership=='Nonowner'].plot(x='Income', y='Lot_Size', style='o',
                                                    markerfacecolor='none', markeredgecolor='C1',
                                                    ax=ax, zorder=100)
    ax.legend(["Owner", "Nonowner"])
    ax.set_xlim(20, 120)
    ax.set_ylim(13, 25)
    ax.set_xlabel('Income ($000s)')
    ax.set_ylabel('Lot size (000s sqft)')
    return ax

fig, ax = plt.subplots(figsize=(7, 4))
ax = basePlot(ax)
plt.tight_layout()
plt.show()

**Cell 25: First split (Income ≤ 59.7)**  
A vertical line at `Income = 59.7` is added to the scatter plot, representing the first split of the decision tree (as seen in the stump earlier). This partition divides the data into two regions, with most owners on the right side and non‑owners on the left.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax = basePlot(ax)
x0 = 59.7
ax.plot((x0, x0), (25, 13), color='grey')
plt.tight_layout()
plt.show()

**Cell 26: Second split (Lot_Size ≤ 21.4 on the left branch)**  
A horizontal line at `Lot_Size = 21.4` is added to the left‑hand region (Income ≤ 59.7). This corresponds to the second split of the depth‑2 tree, further separating owners from non‑owners.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
plt.tight_layout()
plt.show()

**Cell 27: Third split (Lot_Size ≤ 19.8 on the right branch)**  
A horizontal line at `Lot_Size = 19.8` is added to the right‑hand region (Income > 59.7). This is the third split of the depth‑2 tree (or depth‑3 full tree), creating four leaf regions. Each region is now mostly pure (containing one class predominantly).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
y2 = 19.8
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
ax.plot((x0, 120), (y2, y2), color='grey')
plt.tight_layout()
plt.show()

**Cell 28: Further splits (depth‑4 tree illustration)**  
Two additional vertical lines are added (`x3 = 84.75` and `x4 = 61.5`) to further partition the rightmost region. This demonstrates how a deeper tree creates more fine‑grained splits, potentially leading to overfitting if carried too far.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
y2 = 19.8
x3 = 84.75
x4 = 61.5
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
ax.plot((x0, 120), (y2, y2), color='grey')
ax.plot((x3, x3), (13, y2), color='grey')
ax.plot((x4, x4), (13, y2), color='grey')
plt.tight_layout()
plt.show()